# Anomaly Detection on Soil Moisture Sensor Data
## Shallot Cultivation – ESP32 IoT System (6 Sensor Nodes)

**Methods:** Hampel Filter (Statistical) vs Isolation Forest (Machine Learning)

**Pipeline:**
1. Data Loading & Parsing (Firebase JSON)
2. Data Preparation: Missing Values → Z-score Normalization → Time-Series Windowing
3. Synthetic Anomaly Injection: Spike, Drop, Drift, Stuck
4. Anomaly Detection
5. Evaluation: Precision, Recall, F1-Score, ROC-AUC, PR-AUC


---
## 0. Setup & Library Imports

In [ ]:
# ── Google Colab: upload dataset file ──────────────────────────────────
# Run this block ONLY on Google Colab to upload your JSON file.
# Skip if running locally.
import os
IN_COLAB = 'google.colab' in str(__import__('sys').modules)
if IN_COLAB:
    from google.colab import files
    print('Please upload: agriculture-esp32 sunday.json')
    uploaded = files.upload()
    JSON_PATH = list(uploaded.keys())[0]
    print(f'File uploaded: {JSON_PATH}')
else:
    JSON_PATH = r'dataset/dataset json/agriculture-esp32 sunday.json'
    print(f'Local path: {JSON_PATH}')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})
sns.set_palette('tab10')

print('Libraries imported successfully')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  sklearn : {__import__("sklearn").__version__}')

---
## 1. Data Loading & Parsing

Load raw JSON from Firebase / ESP32 with 6 soil moisture sensor nodes.


In [ ]:
with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

records = []
for month_key, month_val in raw_data['agriculture_dataset'].items():
    for day_key, day_val in month_val.items():
        for time_key, entry in day_val.items():
            soil = entry.get('soil', {})
            records.append({
                'recorded_time': pd.to_datetime(entry['recorded_time']),
                'sensor_1': soil.get('1_Yellow_VP'),
                'sensor_2': soil.get('2_Grey_VN'),
                'sensor_3': soil.get('3_Red_D34'),
                'sensor_4': soil.get('4_Green_D35'),
                'sensor_5': soil.get('5_Black_D32'),
                'sensor_6': soil.get('6_Purple_D33'),
            })

df_raw = pd.DataFrame(records).sort_values('recorded_time').reset_index(drop=True)

SENSOR_COLS   = ['sensor_1','sensor_2','sensor_3','sensor_4','sensor_5','sensor_6']
SENSOR_LABELS = ['1-Yellow(VP)','2-Grey(VN)','3-Red(D34)','4-Green(D35)','5-Black(D32)','6-Purple(D33)']

print(f'Total records : {len(df_raw)}')
print(f'Date range    : {df_raw["recorded_time"].min()} -> {df_raw["recorded_time"].max()}')
df_raw.head()

In [ ]:
print('Descriptive Statistics (Raw Soil Moisture, %)')
df_raw[SENSOR_COLS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 14), sharex=True)
colors = plt.cm.tab10.colors

for i, (col, label) in enumerate(zip(SENSOR_COLS, SENSOR_LABELS)):
    axes[i].plot(df_raw['recorded_time'], df_raw[col], color=colors[i], linewidth=0.8, alpha=0.85)
    axes[i].set_ylabel(label, fontsize=8)
    axes[i].set_ylim(0, 110)
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('Raw Soil Moisture Time-Series - All 6 Nodes (ESP32)', fontsize=13, fontweight='bold')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d-%b %H:%M'))
axes[-1].set_xlabel('Recorded Time')
plt.xticks(rotation=30)
fig.tight_layout()
plt.savefig('fig_01_raw_timeseries.png', bbox_inches='tight')
plt.show()
print('Saved: fig_01_raw_timeseries.png')

---
## 2. Data Preparation

Three-step pipeline:
1. **Missing Value Handling** - linear interpolation
2. **Z-score Normalization** - uniform scale per sensor
3. **Time-Series Windowing** - rolling features (mean, std, min, max)


### 2.1 Missing Value Handling

In [ ]:
df = df_raw.set_index('recorded_time')[SENSOR_COLS].resample('1min').mean()
n_gap = df.isnull().sum().sum()
print(f'Implicit gaps after 1-min resample: {n_gap} values')

df.interpolate(method='time', inplace=True)
df.ffill(inplace=True)
df.bfill(inplace=True)

print(f'Missing values after interpolation: {df.isnull().sum().sum()}')
print(f'Total time-steps: {len(df)}')

### 2.2 Z-Score Normalization

In [ ]:
df_clean = df.copy()
df_norm  = df_clean.copy()

for col in SENSOR_COLS:
    mu    = df_norm[col].mean()
    sigma = df_norm[col].std()
    df_norm[col] = (df_norm[col] - mu) / sigma

print('Z-score Normalization Applied')
df_norm.describe().round(3)

### 2.3 Time-Series Windowing

In [ ]:
WINDOW_SIZE = 10  # 10-minute sliding window

def create_window_features(df_input, cols, window=10):
    frames = []
    for col in cols:
        r = df_input[col].rolling(window=window)
        frames.append(pd.DataFrame({
            f'{col}_wmean': r.mean(),
            f'{col}_wstd' : r.std(),
            f'{col}_wmin' : r.min(),
            f'{col}_wmax' : r.max(),
        }))
    out = pd.concat(frames, axis=1).dropna()
    return out

df_features = create_window_features(df_norm, SENSOR_COLS, WINDOW_SIZE)
print(f'Window size   : {WINDOW_SIZE} minutes')
print(f'Feature matrix: {df_features.shape[0]} rows x {df_features.shape[1]} features')
df_features.head(3)

---
## 3. Synthetic Anomaly Injection

| Type | Formula | Description |
|------|---------|-------------|
| **Spike** | x_hat = x_t + delta | Sudden jump (eq.1) |
| **Drop**  | x_hat = x_t - delta | Sudden drop (eq.2) |
| **Drift** | x_hat = x_t + alpha*(t-t0) | Gradual drift (eq.3) |
| **Stuck** | x_hat = c | Sensor frozen (eq.4) |


In [ ]:
np.random.seed(42)

SPIKE_DELTA = 3.5
DROP_DELTA  = 3.5
DRIFT_ALPHA = 0.015
DRIFT_LEN   = 10
STUCK_VALUE = 0.0
STUCK_LEN   = 8
ANOMALY_RATIO = 0.05

df_injected = df_norm.loc[df_features.index].copy()
labels = pd.DataFrame(False, index=df_injected.index, columns=SENSOR_COLS)

n_total = len(df_injected)
points_per = 1 + 1 + DRIFT_LEN + STUCK_LEN
n_per = max(1, int(n_total * ANOMALY_RATIO / points_per))
print(f'n_per_type per sensor: {n_per}')

def inject_anomalies(series, lbl, n_per, spike_d, drop_d, drift_a, drift_len, stuck_val, stuck_len):
    s, l, n = series.copy(), lbl.copy(), len(series)
    # Spike eq.(1)
    sp = np.random.choice(n, size=n_per, replace=False)
    for p in sp: s.iloc[p] += spike_d; l.iloc[p] = True
    # Drop eq.(2)
    dp = np.random.choice(list(set(range(n))-set(sp)), size=n_per, replace=False)
    for p in dp: s.iloc[p] -= drop_d; l.iloc[p] = True
    # Drift eq.(3)
    used = set(sp)|set(dp)
    drs = np.random.choice(list(set(range(n))-used), size=n_per, replace=False)
    for t0 in drs:
        for dt in range(min(t0+drift_len, n)-t0):
            s.iloc[t0+dt] += drift_a*dt; l.iloc[t0+dt] = True
    # Stuck eq.(4)
    au = used|set(drs)
    sts = np.random.choice(list(set(range(n))-au), size=n_per, replace=False)
    for t0 in sts:
        for dt in range(min(t0+stuck_len, n)-t0):
            s.iloc[t0+dt] = stuck_val; l.iloc[t0+dt] = True
    return s, l

for col in SENSOR_COLS:
    df_injected[col], labels[col] = inject_anomalies(
        df_injected[col], labels[col], n_per,
        SPIKE_DELTA, DROP_DELTA, DRIFT_ALPHA, DRIFT_LEN, STUCK_VALUE, STUCK_LEN)

y_true = labels.any(axis=1).astype(int)
print(f'Total time-steps   : {len(y_true)}')
print(f'Anomaly time-steps : {y_true.sum()} ({y_true.mean()*100:.1f}%)')
print()
print('Anomaly count per sensor:')
print(labels.sum().to_frame(name='Anomaly Count'))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_norm.loc[df_features.index, 'sensor_1'],
        color='steelblue', linewidth=0.8, alpha=0.7, label='Clean')
ax.plot(df_injected['sensor_1'],
        color='gray', linewidth=0.6, alpha=0.5, label='With Anomalies')
anom_idx = labels.index[labels['sensor_1']]
ax.scatter(anom_idx, df_injected.loc[anom_idx, 'sensor_1'],
           color='red', s=15, zorder=5, label='Injected Anomalies')
ax.set_title('Sensor 1 - Synthetic Anomaly Injection', fontweight='bold')
ax.set_xlabel('Time'); ax.set_ylabel('Z-score')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('fig_02_anomaly_injection.png', bbox_inches='tight')
plt.show()

---
## 4. Anomaly Detection

### 4.1 Statistical Method: Hampel Filter

- `H_t = |x_t - m| / (1.4826 * MAD)` ... eq.(5)
- `MAD = median(|x_t - m|)` ... eq.(6)
- Flag anomaly when `H_t > k = 3` ... eq.(7)

Each sensor treated **independently**.


In [ ]:
def hampel_filter(series, window_size=10, k=3):
    n = len(series)
    flags  = pd.Series(False, index=series.index)
    scores = pd.Series(0.0,   index=series.index)
    for t in range(n):
        lo = max(0, t - window_size)
        hi = min(n, t + window_size + 1)
        wv  = series.iloc[lo:hi].values
        m   = np.median(wv)
        mad = np.median(np.abs(wv - m))
        H   = 0.0 if mad == 0 else np.abs(series.iloc[t] - m) / (1.4826 * mad)
        scores.iloc[t] = H
        if H > k:
            flags.iloc[t] = True
    return flags, scores

HAMPEL_WINDOW = 10
HAMPEL_K      = 3
hampel_flags, hampel_scores = {}, {}

print('Running Hampel Filter...')
for col in SENSOR_COLS:
    fl, sc = hampel_filter(df_injected[col], HAMPEL_WINDOW, HAMPEL_K)
    hampel_flags[col]  = fl
    hampel_scores[col] = sc
    print(f'  {col}: {fl.sum()} anomalies detected')

y_pred_hamp = pd.DataFrame(hampel_flags).any(axis=1).astype(int)
scores_hamp = pd.DataFrame(hampel_scores).max(axis=1)
print(f'Total Hampel detections: {y_pred_hamp.sum()}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
col = 'sensor_1'
ax1.plot(df_injected.index, df_injected[col], color='steelblue', linewidth=0.8, label='Signal')
det_idx  = hampel_flags[col][hampel_flags[col]].index
ax1.scatter(det_idx, df_injected.loc[det_idx, col], color='orange', s=20, zorder=5, label='Detected')
true_idx = labels.index[labels[col]]
ax1.scatter(true_idx, df_injected.loc[true_idx, col], color='red', marker='x', s=25, zorder=6, label='True Anomaly')
ax1.set_ylabel('Z-score'); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_title('Hampel Filter - Sensor 1', fontweight='bold')
ax2.plot(hampel_scores[col], color='darkorange', linewidth=0.7, label='Hampel Score H_t')
ax2.axhline(HAMPEL_K, color='red', linestyle='--', linewidth=1, label=f'k={HAMPEL_K}')
ax2.set_ylabel('H_t'); ax2.set_xlabel('Time'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('fig_03_hampel_result.png', bbox_inches='tight')
plt.show()

### 4.2 Machine Learning Method: Isolation Forest

`s(x, n) = 2 - E[h(x)] / c(n)` ... eq.(8)

No distribution assumptions. Detects anomalies via random feature splitting.


In [ ]:
df_feat_inj = create_window_features(df_injected, SENSOR_COLS, WINDOW_SIZE)
y_true_if   = y_true.loc[df_feat_inj.index]

# sklearn requires contamination in (0.0, 0.5]
contamination = float(min(y_true_if.mean(), 0.5))
print(f'Ground-truth anomaly rate: {y_true_if.mean():.4f}')
print(f'Contamination used (<=0.5): {contamination:.4f}')

IF_MODEL = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
X = df_feat_inj.values
IF_MODEL.fit(X)

y_pred_if = pd.Series((IF_MODEL.predict(X) == -1).astype(int), index=df_feat_inj.index)
scores_if = pd.Series(-IF_MODEL.decision_function(X),           index=df_feat_inj.index)

print(f'Isolation Forest detected: {y_pred_if.sum()}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax1.plot(df_injected.index, df_injected['sensor_1'], color='steelblue', linewidth=0.8, label='Signal')
if_idx = y_pred_if[y_pred_if == 1].index
ax1.scatter(if_idx, df_injected.loc[if_idx, 'sensor_1'], color='purple', s=15, zorder=5, label='IF Detected')
true_idx = labels.index[labels['sensor_1']]
ax1.scatter(true_idx, df_injected.loc[true_idx, 'sensor_1'], color='red', marker='x', s=25, zorder=6, label='True Anomaly')
ax1.set_ylabel('Z-score'); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
ax1.set_title('Isolation Forest - Detection Result', fontweight='bold')
ax2.fill_between(scores_if.index, scores_if, alpha=0.35, color='purple')
ax2.plot(scores_if, color='purple', linewidth=0.7, label='Anomaly Score')
ax2.axhline(scores_if.quantile(1 - contamination), color='red', linestyle='--', linewidth=1, label='Threshold')
ax2.set_ylabel('Score'); ax2.set_xlabel('Time'); ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('fig_04_iforest_result.png', bbox_inches='tight')
plt.show()

---
## 5. Performance Evaluation

| Metric | Formula |
|--------|---------|
| Precision | TP / (TP+FP) |
| Recall | TP / (TP+FN) |
| F1-Score | 2*(P*R)/(P+R) |
| ROC-AUC | Area under ROC curve |
| PR-AUC | Area under Precision-Recall curve |


In [ ]:
common_idx   = df_feat_inj.index
y_eval       = y_true.loc[common_idx]
y_pred_hamp_ = y_pred_hamp.loc[common_idx]
sc_hamp_     = scores_hamp.loc[common_idx]
y_pred_if_   = y_pred_if.loc[common_idx]
sc_if_       = scores_if.loc[common_idx]

def evaluate(yt, yp, ys, name):
    return {
        'Method'   : name,
        'Precision': round(precision_score(yt, yp, zero_division=0), 4),
        'Recall'   : round(recall_score(yt, yp, zero_division=0), 4),
        'F1-Score' : round(f1_score(yt, yp, zero_division=0), 4),
        'ROC-AUC'  : round(roc_auc_score(yt, ys), 4),
        'PR-AUC'   : round(average_precision_score(yt, ys), 4),
    }

df_results = pd.DataFrame([
    evaluate(y_eval, y_pred_hamp_, sc_hamp_, 'Hampel Filter'),
    evaluate(y_eval, y_pred_if_,   sc_if_,   'Isolation Forest'),
]).set_index('Method')

print('='*55)
print('  PERFORMANCE EVALUATION RESULTS')
print('='*55)
print(df_results.to_string())
print('='*55)

### 5.1 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (yp, title) in zip(axes, [(y_pred_hamp_, 'Hampel Filter'), (y_pred_if_, 'Isolation Forest')]):
    cm = confusion_matrix(y_eval, yp)
    ConfusionMatrixDisplay(cm, display_labels=['Normal','Anomaly']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix - {title}', fontweight='bold')
fig.tight_layout()
plt.savefig('fig_05_confusion_matrices.png', bbox_inches='tight')
plt.show()

### 5.2 ROC-AUC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
RocCurveDisplay.from_predictions(y_eval, sc_hamp_, name='Hampel Filter',    ax=ax, color='darkorange')
RocCurveDisplay.from_predictions(y_eval, sc_if_,   name='Isolation Forest', ax=ax, color='steelblue')
ax.plot([0,1],[0,1],'k--',linewidth=1, label='Random')
ax.set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('fig_06_roc_curves.png', bbox_inches='tight')
plt.show()

### 5.3 Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
PrecisionRecallDisplay.from_predictions(y_eval, sc_hamp_, name='Hampel Filter',    ax=ax, color='darkorange')
PrecisionRecallDisplay.from_predictions(y_eval, sc_if_,   name='Isolation Forest', ax=ax, color='steelblue')
ax.axhline(y_eval.mean(), color='k', linestyle='--', linewidth=1, label=f'Baseline={y_eval.mean():.3f}')
ax.set_title('Precision-Recall Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('fig_07_pr_curves.png', bbox_inches='tight')
plt.show()

### 5.4 Metric Bar Chart Comparison

In [ ]:
metrics = ['Precision','Recall','F1-Score','ROC-AUC','PR-AUC']
x, w = np.arange(len(metrics)), 0.32
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, df_results.loc['Hampel Filter',    metrics], w, label='Hampel Filter',    color='darkorange', alpha=0.85)
b2 = ax.bar(x+w/2, df_results.loc['Isolation Forest', metrics], w, label='Isolation Forest', color='steelblue',  alpha=0.85)
ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.15); ax.set_ylabel('Score')
ax.set_title('Hampel Filter vs Isolation Forest - All Metrics', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
plt.savefig('fig_08_metric_comparison.png', bbox_inches='tight')
plt.show()

---
## 6. Per-Sensor Detection Detail

In [ ]:
rows = []
for col, label_name in zip(SENSOR_COLS, SENSOR_LABELS):
    yt_ = labels[col].astype(int).loc[common_idx]
    yh_ = hampel_flags[col].astype(int).loc[common_idx]
    rows.append({
        'Sensor'   : label_name,
        'Hampel F1': round(f1_score(yt_, yh_,                  zero_division=0), 4),
        'IF F1'    : round(f1_score(yt_, y_pred_if_.loc[common_idx], zero_division=0), 4),
        'Anomalies': int(yt_.sum())
    })
df_per = pd.DataFrame(rows).set_index('Sensor')
print('Per-Sensor F1-Score Breakdown:')
print(df_per.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x, w = np.arange(len(SENSOR_LABELS)), 0.35
b1 = ax.bar(x-w/2, df_per['Hampel F1'], w, label='Hampel Filter',    color='darkorange', alpha=0.85)
b2 = ax.bar(x+w/2, df_per['IF F1'],     w, label='Isolation Forest', color='steelblue',  alpha=0.85)
ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(SENSOR_LABELS, fontsize=9, rotation=20, ha='right')
ax.set_ylim(0, 1.15); ax.set_ylabel('F1-Score')
ax.set_title('Per-Sensor F1-Score: Hampel vs Isolation Forest', fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
plt.savefig('fig_09_per_sensor_f1.png', bbox_inches='tight')
plt.show()

---
## 7. Summary & Export

In [ ]:
wf1  = df_results['F1-Score'].idxmax()
wroc = df_results['ROC-AUC'].idxmax()
wpr  = df_results['PR-AUC'].idxmax()
print('='*62)
print('  ANOMALY DETECTION - FINAL SUMMARY')
print('='*62)
print(f'Dataset     : ESP32 Soil Moisture, 6 sensor nodes')
print(f'Total raw   : {len(df_raw)} records')
print(f'Eval steps  : {len(y_eval)} time-steps')
print(f'Anomaly rate: {y_eval.mean()*100:.2f}%')
print()
print('Best F1-Score :', wf1,  f'({df_results.loc[wf1,  "F1-Score"]:.4f})')
print('Best ROC-AUC  :', wroc, f'({df_results.loc[wroc, "ROC-AUC"]:.4f})')
print('Best PR-AUC   :', wpr,  f'({df_results.loc[wpr,  "PR-AUC"]:.4f})')
print()
print(df_results.to_string())
print('='*62)

In [ ]:
export_df = df_injected.copy()
export_df['is_anomaly']       = y_true.reindex(export_df.index, fill_value=0)
export_df['hampel_pred']      = y_pred_hamp.reindex(export_df.index, fill_value=0)
export_df['hampel_score']     = scores_hamp.reindex(export_df.index, fill_value=0)
export_df['if_pred']          = y_pred_if.reindex(export_df.index, fill_value=0)
export_df['if_score']         = scores_if.reindex(export_df.index, fill_value=0)
export_df.to_csv('soil_moisture_anomaly_labeled.csv')
print('Exported: soil_moisture_anomaly_labeled.csv')
print(f'Shape: {export_df.shape}')
export_df.head()